# 35 · Rollouts with eval gates: test a change offline, ramp it, roll it back
Level L5 · Part 5 · core · ~20 min · builds on lessons 20, 21, 33

Time: about 20 min to read and run. The live run starts the pinned vLLM CPU image **twice** (one server for the stable version, one for the candidate). On the 4-core lab machine the first request of each new server took 10 to 15 minutes in the recorded run, so a live run takes about 30 minutes. Without Docker, the cells print the **recorded output** of a live run (`data/recorded/35.json`), labelled as such. No key is needed. This course runs the engine locally in Docker (the vLLM option in `tools.json`); a GPU VM would use the same flags with a GPU image, which this lesson does not run.

## Goal
By the end you can run an offline eval gate on the real engine: the stable version sets the golden answers, the candidate must pass the gate before any traffic, and a canary with a guard rolls the candidate back by moving its weight to zero.

## Why it matters
- **Without it:** a new config goes to every user at once. If it refuses the long requests, the first sign is a flood of errors in production. The fix is a redeploy of the old version, which takes minutes while the errors continue.
- **With it:** the candidate answers the same fixed prompts as the stable version before any user sees it. A failed check stops the rollout at a small share of traffic, and the rollback is a change of weights to a pool that never stopped.
- **When you don't need it:** a change that cannot change an answer, such as a log line, needs only the normal deploy checks. A change to the model, its flags, its quantization or its prompt needs the gate.

## The idea
A rollout moves traffic from the **stable** version to a **candidate** in steps. Before any user sees the candidate, an **eval gate** runs both versions on the same fixed prompts. The stable answers are the **golden** answers. The gate checks the candidate's error rate and how often its answers match the golden ones. If the gate passes, a **canary** sends a small share of traffic to the candidate, and a guard reads its errors at each step. **Rollback** is a change of weights, not a redeploy, because the stable pool never stopped. It is like a kitchen that tests a new recipe on a few tables while the old recipe stays on the menu, and switches back in one step when the plates come back.

## Picture
```mermaid
flowchart LR
  E["Fixed eval set: 20 cases"] --> S["Stable version: golden answers"]
  E --> C["Candidate: same cases"]
  S --> G{"Offline gate"}
  C --> G
  G -- "fail" --> K["Stable keeps 100% of traffic"]
  G -- "pass" --> R["Canary: 1%, 5%, 25%, 100%"]
  R -- "guard sees errors" --> K
```

## Step by step

### Step 1: the change under test and the eval set
The candidate changes one flag: the context limit goes from 2048 to 1600 tokens. A team might make that change to cut costs and not test the long requests. The eval set has 16 short support questions and 4 long messages of 250 random support words. The policy text is made up, and it is the same seeded text as lesson 18.

In [1]:
import asyncio, json, os, random, shutil, subprocess, sys, time
from pathlib import Path
import httpx
sys.path.insert(0, "..")                                   # the course folder, so tut_env.py can be imported
from tut_env import env

NAME, PORT = "tut-kv-forge-15h-35", 8350                   # this lesson's container and port
BASE = f"http://127.0.0.1:{PORT}"
IMAGE, SERVED = "vllm/vllm-openai-cpu:v0.32.0", "brightside-qwen-0.5b"    # pinned, as in lessons 08 to 33
KEY = env("VLLM_API_KEY", "")                               # empty when unset; the value is never printed
HEADERS = {"Authorization": f"Bearer {KEY}"} if KEY else {}
COMMON = ["Qwen/Qwen2.5-0.5B-Instruct", "--served-model-name", SERVED, "--host", "0.0.0.0", "--dtype", "float32",
          "--enforce-eager", "--max-num-seqs", "4", "--max-num-batched-tokens", "2048",
          "--kv-cache-memory-bytes", str(2 * 1024**3)]
STABLE = COMMON + ["--max-model-len", "2048"]
CANDIDATE = COMMON + ["--max-model-len", "1600"]            # the change under test
print("flags that differ (stable, candidate):", [(a, b) for a, b in zip(STABLE, CANDIDATE) if a != b])
RUN = {}                                                   # live results go here; empty means: use the recording

flags that differ (stable, candidate): [('2048', '1600')]


The policy and the 20 cases. The policy is the seeded text of lesson 18, so its tokens match that lesson.

In [2]:
TOPICS = ["refund", "delivery", "warranty", "password reset", "billing date", "gift card", "exchange"]
rng = random.Random(18)                                    # the same made-up policy as lesson 18
POLICY = "\n".join(f"Rule {i}: for a {rng.choice(TOPICS)} request, check the order date, confirm the account "
                   f"owner, and answer within {rng.randint(1, 10)} business days." for i in range(1, 51))
SHORT = ["Can I return a gift card?", "Where is my parcel?", "My password reset email is late.",
         "Can I change my billing date?", "The exchange item arrived damaged.", "How long does a warranty claim take?",
         "Can I get a refund without the receipt?", "Why was my account locked?", "Do gift cards expire?",
         "Can I cancel a delivery?", "Who can open my account?", "Is a warranty claim free?",
         "How do I get a refund to my card?", "Can I split a payment?", "When is my billing date?",
         "Can I change my delivery address?"]
WORDS = "order refund parcel damaged email wait team case account card delivery warranty".split()
rng35 = random.Random(35)
LONG = [" ".join(rng35.choice(WORDS) for _ in range(250)) + " What should I do next?" for _ in range(4)]
CASES = [{"id": i, "kind": "short", "question": q} for i, q in enumerate(SHORT)] + \
        [{"id": 16 + i, "kind": "long", "question": q} for i, q in enumerate(LONG)]
print(f"{len(CASES)} cases: {len(SHORT)} short, {len(LONG)} long | policy: {len(POLICY.split())} made-up words")

20 cases: 16 short, 4 long | policy: 1023 made-up words


### Step 2: the helpers
`start` runs the pinned image with one set of flags and waits for `/health`. Requests use greedy decoding (temperature 0), so the same input gives the same answer.

In [3]:
def server_up():
    try:
        return httpx.get(f"{BASE}/health", timeout=2).status_code == 200
    except httpx.TransportError:
        return False

def start(flags):
    """Start the pinned engine with `flags` and wait until /health answers."""
    subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
    run_env, key_flag = dict(os.environ), []
    if KEY:
        run_env["VLLM_API_KEY"], key_flag = KEY, ["-e", "VLLM_API_KEY"]
    cmd = ["docker", "run", "-d", "--name", NAME, "-p", f"{PORT}:8000", "--shm-size", "4g",
           "-v", f"{Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface", *key_flag, IMAGE, *flags]
    if subprocess.run(cmd, capture_output=True, text=True, env=run_env, timeout=120).returncode != 0:
        raise RuntimeError("docker run failed")
    deadline = time.time() + 900
    while time.time() < deadline and not server_up():
        time.sleep(3)
    if not server_up():
        raise TimeoutError("server not healthy after 900 s")

def stop():
    subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)

`ask` sends one case. `run_eval` sends all cases with four in flight, which matches `--max-num-seqs 4`. A refused request is kept with its HTTP status and error text.

In [4]:
async def ask(client, case, sem):
    body = {"model": SERVED, "temperature": 0, "max_tokens": 48,
            "messages": [{"role": "system", "content": POLICY}, {"role": "user", "content": case["question"]}]}
    async with sem:
        r = await client.post(f"{BASE}/v1/chat/completions", json=body, headers=HEADERS, timeout=900)
    out = {"id": case["id"], "kind": case["kind"], "status": r.status_code}
    if r.status_code == 200:
        d = r.json()
        out |= {"answer": d["choices"][0]["message"]["content"], "prompt_tokens": d["usage"]["prompt_tokens"]}
    else:
        out["error"] = r.text[:200]
    return out

async def run_eval():
    sem = asyncio.Semaphore(4)
    async with httpx.AsyncClient() as client:
        return await asyncio.gather(*(ask(client, c, sem) for c in CASES))

### Step 3: the stable version sets the golden answers
Start the stable configuration, send the 20 cases, and stop it. Its answers are the golden set for the gate. The first request of the new server takes 10 to 15 minutes (a one-time warm-up). The live cells run only with `TUT_ENABLE=docker` on a machine with Docker. Otherwise they are skipped, `RUN` stays empty, and Step 5 reads the recording (its output says "recorded output").

In [5]:
if shutil.which("docker"):
    start(STABLE)
    try:
        RUN["stable"] = {"flags": STABLE, "results": await run_eval()}
    finally:
        stop()
    print("stable: cases", len(RUN["stable"]["results"]), "| refused:", sum(r["status"] != 200 for r in RUN["stable"]["results"]))
else:
    print("Docker not found: the recorded run is used in Step 5")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 4: the candidate runs the same cases
Start the candidate with the one changed flag, send the same 20 cases, and stop it. Only one engine runs at a time here, to keep the lab machine's 15 GB of memory free. In production both pools run at once (see At work).

In [6]:
if shutil.which("docker"):
    start(CANDIDATE)
    try:
        RUN["candidate"] = {"flags": CANDIDATE, "results": await run_eval()}
    finally:
        stop()
    print("candidate: cases", len(RUN["candidate"]["results"]), "| refused:", sum(r["status"] != 200 for r in RUN["candidate"]["results"]))
else:
    print("Docker not found: the recorded run is used in Step 5")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


The live results are saved as a run-time file (not committed). The recording in `data/recorded/35.json` is a copy of this run.

In [7]:
if RUN.get("candidate"):
    Path("../data/35").mkdir(parents=True, exist_ok=True)
    Path("../data/35/results.json").write_text(json.dumps(RUN, indent=1))
    print("saved the live run to data/35/results.json (run-time file, not committed)")
else:
    print("no live run to save: the recording is used in Step 5")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 5: the gate
The gate has two rules, both written before the run. The first is an error budget: the **lower** end of the 95% Wilson interval for the candidate's error rate must stay at or below 1%. With 20 cases, an error count can be noisy, so the interval decides, not the point value. The second is agreement: at least 95% of the candidate's served answers must match the golden answer. The stable pool must also serve every case, as a sanity check.

`wilson` gives the rate and its 95% interval (lesson 21). It stays inside 0 to 100% at the extremes.

In [8]:
def wilson(passes, n, z=1.96):
    """Rate and its 95% Wilson interval (it stays inside 0 to 1 at 0% and 100%)."""
    p = passes / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / (1 + z * z / n)
    return p, centre - half, centre + half

`gate` applies the two rules to the two runs and prints each check, then the decision. The agreement check compares the strings exactly. That is acceptable here because both runs use temperature 0, the same image and the same model, so the same input gives the same text. With sampling, or a different model, compare with a similarity score or a judge, and set the threshold from the measured noise.

In [9]:
MAX_ERROR_RATE, MIN_AGREEMENT = 0.01, 0.95               # the two rules of the gate
print("gate rules: error lower bound <=", MAX_ERROR_RATE, "| agreement >=", MIN_AGREEMENT)

def gate(stable, cand):
    gold = {r["id"]: r.get("answer") for r in stable}
    served = [r for r in cand if r["status"] == 200]
    errors = len(cand) - len(served)
    p, lo, hi = wilson(errors, len(cand))
    agree = sum(r["answer"] == gold[r["id"]] for r in served) / len(served) if served else 0.0
    checks = {"stable pool served every case": all(r["status"] == 200 for r in stable),
              "error rate, lower 95% bound at most 1%": lo <= MAX_ERROR_RATE,
              "served answers match golden at least 95%": agree >= MIN_AGREEMENT}
    print(f"candidate errors: {errors} of {len(cand)} ({p:.0%}, 95% CI {lo:.0%} to {hi:.0%}); "
          f"served answers matching golden: {agree:.0%} of {len(served)}")
    for name, ok in checks.items():
        print(f"  {'pass' if ok else 'FAIL'}: {name}")
    return "promote" if all(checks.values()) else "roll back"

R = RUN if RUN.get("candidate") else json.loads(Path("../data/recorded/35.json").read_text())
print("source:", "this live run" if RUN.get("candidate") else "recorded output (no live run here)")
print("candidate flags differ from stable by:", [(a, b) for a, b in zip(R["stable"]["flags"], R["candidate"]["flags"]) if a != b])
DECISION = gate(R["stable"]["results"], R["candidate"]["results"])
print("decision:", DECISION)
for case_id in (0, 16):                                    # one short and one long case, from the stable run
    r = next(x for x in R["stable"]["results"] if x["id"] == case_id)
    print(f"stable prompt tokens, case {case_id} ({r['kind']}): {r['prompt_tokens']} (context limit 1600)")

gate rules: error lower bound <= 0.01 | agreement >= 0.95
source: recorded output (no live run here)
candidate flags differ from stable by: [('2048', '1600')]
candidate errors: 4 of 20 (20%, 95% CI 8% to 42%); served answers matching golden: 100% of 16
  pass: stable pool served every case
  FAIL: error rate, lower 95% bound at most 1%
  pass: served answers match golden at least 95%
decision: roll back
stable prompt tokens, case 0 (short): 1436 (context limit 1600)
stable prompt tokens, case 16 (long): 1685 (context limit 1600)


**Read it.** Every served answer matches golden, so the agreement rule passes. The error rule fails: the long cases need more than 1,600 tokens (the stable run's long prompt is 1,685 tokens, a short one 1,436), and the engine refuses them with HTTP 400. Four errors in 20 is a 20% rate, and its interval starts far above 1%. The gate says "roll back" and the rollout never starts. The average over the served answers hides the problem; the error count shows it.

### Step 6: the canary, if the gate had passed
Suppose the gate had passed. A canary sends 1%, 5%, 25% and then 100% of traffic to the candidate. The guard reads a step after 10 candidate answers and rolls back if the step's raw error rate is above 1%. It uses the raw rate, not the Wilson bound: one error in 10 is already 10%, so a step survives only if all 10 succeed. That is stricter and noisier than the gate, on purpose. The traffic is **simulated** from the measured error rate, which assumes the live mix matches the eval set. Real traffic has fewer long messages, so its real rate is lower; that is why the gate runs first.

`Router` is the gateway's split between two pools. A rollback changes the weights; no pool restarts.

In [10]:
class Router:
    """The gateway's split between two pools. A rollback changes the weights; no pool is restarted."""
    def __init__(self):
        self.weight = {"stable": 1.0, "candidate": 0.0}
    def set_candidate(self, w):
        self.weight = {"stable": round(1 - w, 4), "candidate": w}
    def rollback(self):                                    # the stable pool never stopped: nothing to wait for
        self.weight = {"stable": 1.0, "candidate": 0.0}
    def pick(self, rng):
        return rng.choices(list(self.weight), weights=list(self.weight.values()))[0]

`ramp` runs the four steps. Each step waits for 10 candidate answers, then the guard reads the errors.

In [11]:
def ramp(error_rate, steps=(0.01, 0.05, 0.25, 1.0), min_sample=10, seed=35):
    """Four steps; each waits for 10 candidate answers, then the guard reads its errors."""
    rng, router, rows, total, exposed = random.Random(seed), Router(), [], 0, 0
    for w in steps:
        router.set_candidate(w)
        req = seen = errs = 0
        while seen < min_sample:
            req += 1
            if router.pick(rng) == "candidate":
                seen += 1
                errs += rng.random() < error_rate
        total, exposed = total + req, exposed + errs
        verdict = "roll back" if errs / seen > MAX_ERROR_RATE else "next step"
        rows.append((f"{w:.0%}", req, seen, errs, verdict))
        if verdict == "roll back":
            router.rollback()
            break
    return rows, router, exposed, total

Expect only one row: the guard rolls back at the first step, so the later steps never run. Collecting 10 candidate answers at weight *w* takes about 10 ÷ *w* requests on average, so about 1,000 at 1%. The run below is one seeded draw (`seed=35`), and it needed 610. Run the ramp with the error rate measured in Step 5 (4 of 20 here), and print each step.

In [12]:
ERROR_RATE = sum(r["status"] != 200 for r in R["candidate"]["results"]) / len(CASES)   # measured in Step 5
rows, router, exposed, total = ramp(ERROR_RATE)
print("SIMULATION over the measured error rate (no engine running)")
print(f"{'step':>5} {'requests':>9} {'to candidate':>13} {'errors':>7}  verdict")
for w, req, seen, errs, verdict in rows:
    print(f"{w:>5} {req:>9} {seen:>13} {errs:>7}  {verdict}")
print("final weights:", router.weight, "| simulated requests that failed:", exposed, "of", total)
print(f"a big-bang to 100% over the same traffic would send about {round(total * ERROR_RATE)} errors ({ERROR_RATE:.0%} measured rate)")

SIMULATION over the measured error rate (no engine running)
 step  requests  to candidate  errors  verdict
   1%       610            10       3  roll back
final weights: {'stable': 1.0, 'candidate': 0.0} | simulated requests that failed: 3 of 610
a big-bang to 100% over the same traffic would send about 122 errors (20% measured rate)


**Read it.** The numbers are one seeded draw, not a live fact. At 1% of traffic the candidate needed 610 requests to collect its 10 answers. In that step 3 of the 10 failed, so the guard rolled back, and the weights went back to 100% stable in one step. Three simulated requests failed. The big-bang comparison uses the same 20% rate, so the gap is the ramp's blast radius: about 122 errors instead of 3.

## At work
- **Silent quality loss (RESEARCH.md, section 7, row 11):** a "free" change, such as a quantization, a KV-cache setting or an engine upgrade, can lower quality with no error. Teams gate it on a golden set. → lesson 21 (the regression set and its error bars; same idea for KV quantization).
- **Standard tests miss production failures (section 4, Anthropic postmortem, 2025):** a routing change and a runtime bug degraded answers while benchmarks and spot checks missed it. The team added quality evals on production traffic. → lesson 37 (read the metrics dashboard that would show a routing or runtime fault).
- **Two pools cost capacity (section 8.3):** during a canary the fleet runs the stable pool and the candidate at once. Size headroom for that and for a lost node first. → lesson 38 (capacity and HA/DR maths).
- **Go deeper:** the vLLM docs describe how each engine flag changes behaviour; check the flag and its default in the pinned version before a rollout.

## Common mistakes
**1. Reading an answer from a refused request.** A 400 response has no `answer`. Filtering on the status first avoids the crash. The cell below reads every candidate answer and fails on the first refused case. **This cell fails on purpose.**

In [13]:
answers = {r["id"]: r["answer"] for r in R["candidate"]["results"]}
print(len(answers), "answers")

KeyError: 'answer'

**The fix:** keep only the rows with `status == 200`, as `gate()` does: `{r["id"]: r["answer"] for r in cand if r["status"] == 200}`.

**2. Using the candidate's own answers as golden.** A candidate compared with itself always agrees. The golden set must come from the version that is live now, and it must be saved before the candidate starts.

**3. Rolling back by redeploying the old image.** A redeploy restarts the pool, and the errors continue for minutes. Keep the stable pool running until the candidate has passed every step, then move the weight back.

## Production checklist
- Save the golden answers from the live version before the candidate starts, and version the eval set with them.
- Gate on the error count with its interval, and on agreement with golden on the served answers. A mean over served answers hides refusals.
- Change one thing at a time, so a failure has one cause (here, one flag).
- Keep the stable pool up until the candidate passes every canary step. Rollback is a weight change.
- Set the minimum sample before a guard reads a step (here 10 candidate answers), so one lucky request does not decide.
- Drain before you stop a pool: stop new routing, let in-flight streams finish, then send SIGTERM. Set the termination grace period above your longest generation (lesson 33).
- Size the fleet for two pools during the canary.

## Try it
From `R`, compute the candidate's error rate for the short cases and for the long cases separately, with the 95% Wilson interval for each. Which group does the error budget catch, and why does the agreement rule alone pass?

In [14]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [15]:
for kind in ("short", "long"):
    grp = [r for r in R["candidate"]["results"] if r["kind"] == kind]
    errs = sum(r["status"] != 200 for r in grp)
    p, lo, hi = wilson(errs, len(grp))
    print(f"{kind:5} {errs} of {len(grp)} refused ({p:.0%}, 95% CI {lo:.0%} to {hi:.0%})")
print("the long cases are caught by the error budget; the agreement rule only sees the served short answers, which match golden")

short 0 of 16 refused (0%, 95% CI 0% to 19%)
long  4 of 4 refused (100%, 95% CI 51% to 100%)
the long cases are caught by the error budget; the agreement rule only sees the served short answers, which match golden


## Recap
- An eval gate runs the candidate on the same fixed cases as the stable version, with golden answers from the live version, before any traffic.
- Check the error rate with its interval, and agreement only on served answers. The mean of served answers alone hides refusals.
- A canary moves weight in steps, and its guard rolls back by moving the weight back to the stable pool, which never stopped.
- Drain (stop new routing and let in-flight requests finish) before a pool stops, and keep headroom for two pools during the change.
- **Interview:** "How do you roll out a new model to an LLM fleet without downtime, and know it is not worse?" Strong answer: keep the stable pool up; gate the candidate offline on golden answers, checking the error budget at its confidence bound; ramp by weight with a guard that rolls back by weight; drain before a pool stops.